# H&M M5-C 마지막 1주 작업 평가 — 1차 계산 배치

- 학습: 2018-09-20~2020-09-15 전체 / test: 2020-09-16~22 / validation·holdout 없음
- 최종 계획은 seed 42~51, 고정 300 epoch, M1·M3·M4-C·M5-C·공동순열 M5-C 50개 arm입니다.
- **이 노트북은 1차로 seed 49·50의 M1·M3·M5-C 6개 arm만 계산**합니다.
- 1차 수치는 중간 계산 점검용이며, 성공·실패 판정이나 설정 재조정에 쓰지 않습니다.
- 중단 후 실행 셀을 다시 누르면 저장된 epoch에서 재개합니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'GPU 런타임(L4 권장)을 연결하세요.'
SOURCE_COMMIT = 'c62e861'
REPO = Path('/content/clv-m5c-hm-final10-' + SOURCE_COMMIT)
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
subprocess.run(['git','-C',str(REPO),'checkout',SOURCE_COMMIT], check=True)
sys.path.insert(0, str(REPO))
print('GPU:', torch.cuda.get_device_name(0), '| code:', SOURCE_COMMIT)

In [ ]:
import pandas as pd
import clv_m5_user_centered_hm2y_final10 as screen

cfg = screen.configure()
print(json.dumps(screen.preflight_summary(cfg), ensure_ascii=False, indent=2))
prepared = screen.prepare(cfg)
stats = prepared['data']['data_stats']
bounds = stats['split_boundaries']
assert set(prepared['data']['splits']) == {'test'}
assert stats['split_rows']['val'] == 0 and stats['split_rows']['holdout'] == 0
assert pd.Timestamp(bounds['train']['end_inclusive']) == pd.Timestamp('2020-09-15')
assert pd.Timestamp(bounds['test']['start_exclusive']) == pd.Timestamp('2020-09-15')
assert pd.Timestamp(bounds['test']['end_inclusive']) == pd.Timestamp('2020-09-22')
assert prepared['data']['loss_w'] is None
print('사전점검 완료 — 마지막 1주 test·검증/holdout 없음·고정 300 epoch입니다.')

In [ ]:
# 1차 계산 배치: seed 49·50 × M1·M3·M5-C = 6 arm
result = screen.run_batch(
    prepared, cfg,
    seeds=screen.INITIAL_SEEDS,
    model_ids=screen.CORE_MODELS,
)
print(json.dumps(result['status'], ensure_ascii=False, indent=2))
assert result['status']['initial_batch_complete']
assert result['status']['final_decision_available'] is False
assert result['status']['retuning_from_partial_results_allowed'] is False

In [ ]:
from IPython.display import display
metrics = ['recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
           'price_purchase_amount_weighted_hit@10','vndcg@10']
print('※ 아래는 6/50 arm 중간 수치이며 최종 성공·실패 판정이 아닙니다.')
display(result['absolute'][['model_id','seed',*metrics]].sort_values(['seed','model_id']))

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

output = Path('/content/clv_m5_user_centered_hm2y_final10_batch1_seed49_50_results.zip')
with ZipFile(output, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, Path(path).name)
    archive.write(prepared['run_dir'] / 'protocol.json', 'protocol.json')
files.download(str(output))